# Explore the SEC filings agent

Interactive playground for everything Phase 1 built. **Edit the strings and re-run cells.**

**Setup (once):** select the project's `.venv` as the kernel (top-right kernel picker → it's at `.venv\Scripts\python.exe`). Everything imports from `src/sec_filings/`; no logic lives here.

What you can do below:
1. Ask the agent any question → see the answer
2. Inspect the agent's trace → see how it routed and chained tools
3. Run raw hybrid retrieval → see the actual chunks that come back
4. Hit the FMP numeric tool directly
5. See what's indexed (the corpus manifest)

In [ ]:
from sec_filings.agent.loop import run_agent
from sec_filings.retrieval.hybrid import hybrid_search
from sec_filings.tools.fmp import get_financial_fact, SUPPORTED_CONCEPTS
from sec_filings.config import settings

import mistune
from IPython.display import HTML, display


def show_md(text, height=420):
    """Render markdown text (e.g. an agent answer) as a scrollable HTML box."""
    html = mistune.html(text)
    display(HTML(
        f'<div style="max-height:{height}px;overflow:auto;padding:10px 14px;'
        f'border:1px solid #ddd;border-radius:8px;line-height:1.55">{html}</div>'
    ))


print("agent model :", settings.agent_model)
print("embed model :", settings.embed_model)
print("FMP concepts:", sorted(SUPPORTED_CONCEPTS))

## 1. Ask the agent anything

The agent routes between `retrieve` (filing prose) and `get_financial_fact` (exact numbers) on its own. Only MSFT FY2023 is indexed right now, so keep questions to that filing (it'll tell you if you stray).

**This is genuinely agentic, so it's not instant.** A simple question is one tool call (~5s); a compound one (like the default below) makes several Claude round-trips + tool calls (~20–40s). The `live` hook prints each step as it happens — if you see tool calls printing, it's working, not stalled.

Try swapping in your own question.

In [ ]:
QUESTION = "What was Microsoft's gross margin in fiscal 2023, and how does it describe its cloud strategy?"


# A compound question is genuinely multi-step: the agent makes several Claude
# round-trips + tool calls, so it can take 20-40s. on_event prints each step the
# moment it happens, so you can WATCH it work instead of staring at a blank cell.
def live(step):
    if step.type == "tool_call":
        print(f"  -> {step.tool_name}({step.tool_input})", flush=True)
    elif step.type == "tool_result":
        print(f"  <- {step.tool_name} returned", flush=True)
    elif step.type == "answer":
        print("  done.\n", flush=True)


print(f"Q: {QUESTION}\n")
trace = run_agent(QUESTION, on_event=live)
show_md(trace.final_answer)

## 1b. Open this run as a Langfuse trace

If `LANGFUSE_*` is set in `.env`, every `run_agent(...)` call is also a **waterfall trace on the Langfuse website**: a root `agent` span → one `generation` per Claude turn (with token usage + auto-computed USD cost) → a `retriever` span for hybrid search and a `tool` span for the FMP numeric tool. Run the cell below to jump straight to it.

In [ ]:
# The trace was already flushed to Langfuse when run_agent returned above.
from sec_filings import observability as obs

url = obs.trace_url(trace.langfuse_trace_id)
print(url or "Langfuse not configured — set LANGFUSE_* in .env to enable tracing.")

## 2. Inspect the trace

This is the agentic part — every reasoning blurb, tool call (with its inputs), and tool result, in order. This is how you *see* the routing and chaining, not just the final answer.

In [ ]:
def show_trace(trace):
    print(f"Q: {trace.question}")
    print(f"model={trace.model}  tokens in/out={trace.total_tokens_in}/{trace.total_tokens_out}  "
          f"elapsed={trace.elapsed_seconds:.1f}s  steps={len(trace.steps)}")
    print("-" * 80)
    for s in trace.steps:
        if s.type == "tool_call":
            print(f"[{s.step_index}] TOOL_CALL  {s.tool_name}({s.tool_input})")
        elif s.type == "tool_result":
            preview = str(s.tool_output)
            preview = preview if len(preview) <= 200 else preview[:200] + "..."
            print(f"[{s.step_index}] TOOL_RESULT {s.tool_name} -> {preview}")
        elif s.type == "reasoning":
            print(f"[{s.step_index}] REASONING  {s.content[:200]}")
        elif s.type == "answer":
            print(f"[{s.step_index}] ANSWER     ({len(s.content)} chars)")


show_trace(trace)

## 3. Raw hybrid retrieval

Skip the agent and look straight at what the retriever returns for a query — the fused (dense + BM25) top-k chunks, with which ranker surfaced each one. This is the cell to poke at when retrieval feels off.

In [ ]:
hits = hybrid_search("how does management discuss artificial intelligence", ticker="MSFT", fiscal_year=2023, top_k=5)

for h in hits:
    print(f"score={h.fused_score:.4f}  dense_rank={h.dense_rank}  lexical_rank={h.lexical_rank}  "
          f"[{h.chunk.item}]")
    print(f"  {h.chunk.text[:300].strip()!r}")
    print()

## 4. FMP numeric tool, directly

The structured-data backend. Returns exact reported figures (no LLM, no parsing risk).

In [ ]:
rev = get_financial_fact("MSFT", 2023, "revenue")
rnd = get_financial_fact("MSFT", 2023, "researchAndDevelopmentExpenses")
print(f"revenue: ${rev:,.0f}")
print(f"R&D:     ${rnd:,.0f}")
print(f"R&D as % of revenue: {rnd / rev:.2%}")

## 5. What's indexed

The manifest is the source of truth for what the retriever can answer over (Phase 2 will fill this out).

In [ ]:
import json

manifest = json.loads((settings.data_dir / "chroma_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))